# Kirundi news topic classification: Transformer fine-tuning on Google Colab
**Before running:** Runtime > Change runtime type > **T4 GPU**.
In Google Drive create a folder `kirnews-processed` and upload the 6 files from your local `data/processed/`
(`raw_train.csv`, `raw_val.csv`, `raw_test.csv`, `raw_official_train.csv`, `raw_official_test.csv`, `raw_report.json`).
Results are saved to Drive (`kirnews-results/`), so a disconnected session does not lose finished runs; just re-run the cells and finished runs are skipped.

In [ ]:
import os, subprocess, torch
from google.colab import drive
drive.mount('/content/drive')
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - change the runtime type")
DATA_DIR = "/content/drive/MyDrive/kirnews-processed"
RESULTS_DRIVE = "/content/drive/MyDrive/kirnews-results/transformers"
print(os.listdir(DATA_DIR))

In [ ]:
REPO = "https://github.com/Delucie-Rurangwa/kirundi-news-classification.git"
if not os.path.exists("/content/kirundi-news-classification"):
    subprocess.run(["git", "clone", REPO], check=True, cwd="/content")
os.chdir("/content/kirundi-news-classification")
subprocess.run(["git", "pull"], check=True)
# send per-run results straight to Drive so they survive a disconnect
os.makedirs(RESULTS_DRIVE, exist_ok=True)
os.makedirs("results", exist_ok=True)
if not os.path.exists("results/transformers"):
    os.symlink(RESULTS_DRIVE, "results/transformers")
print(os.listdir("results"))

In [ ]:
!pip -q install transformers accelerate tabulate

## 1. Smoke test (1-2 minutes). It must finish without errors before the long runs.

In [ ]:
!python src/finetune_transformer.py --model Davlan/afro-xlmr-base --data_dir {DATA_DIR} --out_dir /content/smoke --smoke --name smoke_s42

## 2. Experiments (4 configurations x 3 seeds = 12 runs)
| ID | Model | Max length | Loss | Question it answers |
|---|---|---|---|---|
| E1 | `xlm-roberta-base` | 256 | plain CE | How does a general multilingual model do? |
| E2 | `Davlan/afro-xlmr-base` | 256 | plain CE | Does African-language pretraining help? |
| E3 | `Davlan/afro-xlmr-base` | 256 | class-weighted CE | Does weighting help the rare classes? |
| E4 | `Davlan/afro-xlmr-base` | 512 | class-weighted CE | Does reading more of the article help? |

In [ ]:
SEEDS = [42, 43, 44]
EXPS = {
 "E1_xlmr_base_len256":            dict(model="xlm-roberta-base",      max_len=256, weighted=False, batch=16, accum=1),
 "E2_afroxlmr_base_len256":        dict(model="Davlan/afro-xlmr-base", max_len=256, weighted=False, batch=16, accum=1),
 "E3_afroxlmr_base_len256_wloss":  dict(model="Davlan/afro-xlmr-base", max_len=256, weighted=True,  batch=16, accum=1),
 "E4_afroxlmr_base_len512_wloss":  dict(model="Davlan/afro-xlmr-base", max_len=512, weighted=True,  batch=8,  accum=2),
}
for name, c in EXPS.items():
    for seed in SEEDS:
        run = f"{name}_s{seed}"
        if os.path.exists(f"results/transformers/result_{run}.json"):
            print("skip (done):", run); continue
        cmd = ["python", "src/finetune_transformer.py", "--model", c["model"], "--data_dir", DATA_DIR,
               "--max_len", str(c["max_len"]), "--batch", str(c["batch"]), "--grad_accum", str(c["accum"]),
               "--seed", str(seed), "--name", run]
        if c["weighted"]: cmd.append("--weighted_loss")
        print("RUNNING", run, flush=True)
        subprocess.run(cmd, check=True)

## 3. Aggregate into the experiments table

In [ ]:
!python src/aggregate_results.py

## 4. Train and save the final model for the web app (edit flags to match the winning config)

In [ ]:
!python src/finetune_transformer.py --model Davlan/afro-xlmr-base --data_dir {DATA_DIR} --max_len 256 --weighted_loss --seed 42 --name final_s42 --out_dir /content/final_results --save_dir /content/final_model
!mkdir -p /content/drive/MyDrive/kirnews-results && cp -r /content/final_model /content/drive/MyDrive/kirnews-results/final_model && cp /content/kirundi-news-classification/results/all_runs.csv /content/kirundi-news-classification/results/experiments_table.csv /content/drive/MyDrive/kirnews-results/